# Original Question: Design Persistent Key-Value Store

+1 more  
Hard  
Hash Table

Interview Stages  
Onsite  
Frequency  
Asked By  
OPENAI-icon  
Last Reported  
1 weeks ago

## AI Insights

You are designing a persistent key-value store that supports saving and restoring its data using a binary middleware medium. The system should store only string keys and string values and be able to serialize its entire state to a binary blob for durability across restarts.

A storage interface called Medium is provided, allowing you to save and retrieve a single binary blob. The implementation is in InMemoryMedium.

```java
// This is a provided API interface.
interface Medium {
    /**
     * Saves a byte array to the storage medium (e.g., file, in-memory).
     * @param data The byte array to persist.
     */
    void saveBlob(byte[] data);

    /**
     * Retrieves the most recently saved byte array from the storage medium.
     * @return The stored byte array, or null if nothing is saved.
     */
    byte[] getBlob();
}
```

You need to implement the KVStore class, ensuring that the binary serialization format can exactly recover all string keys and values, including those with Unicode characters or empty strings. Native object serialization (such as direct use of language-specific serialization, or JSON) is not allowed. You must construct your own serialization format.

## Implement the KVStore class

- `KVStore(Medium medium)` Initializes a new, empty key-value store using the provided storage medium. The store only interacts with data via this medium.
- `void put(String key, String value)` Stores the specified value associated with the given key. If the key already exists, its value is updated.
- `String get(String key)` Returns the value associated with key. If the key is not present, it returns the empty string `""`.
- `void shutdown()` Serializes the current store into a custom binary format and writes it to the associated medium. After shutdown, the instance becomes closed, any subsequent get or put must fail and throw an error.
- `void restore()` Loads the key-value data from the binary blob saved in the medium, completely replacing the in-memory store. If the blob is missing or empty, the store should become empty.

## Constraints

- Keys and values are arbitrary strings (may include Unicode and empty strings).
- After calling shutdown, the instance is permanently closed; further calls to get or put must fail.
- The restore method always replaces the in-memory state, discarding any unsaved changes.
- 1 ≤ number of keys ≤ 10⁴.
- Total size of all keys and values together ≤ 10⁶ characters.
- Each key and value is a string of length ≤ 10⁴.
- All methods are called in a valid order (e.g., shutdown is not called twice in a row).

## Example

**Input:**

```text
["InMemoryMedium", "KVStore", "put", "put", "put", "get", "get", "shutdown", "KVStore", "get", "put", "restore", "get", "get", "get", "get"]
[[], [], ["key1", "value1"], ["key2", "value2_is_longer"], ["key3", "value3"], ["key1"], ["key4"], [], [medium], ["key1"], ["key4", "value4"], [], ["key1"], ["key2"], ["key3"], ["key4"]]
```

**Output:**

```text
[null, null, null, null, null, "value1", "", null, null, "", null, null, "value1", "value2_is_longer", "value3", ""]
```

**Explanation:**

```java
Medium medium = new InMemoryMedium();
KVStore store1 = new KVStore(medium); // Initializes a new store backed by medium.
store1.put("key1", "value1"); // Stores "key1" -> "value1".
store1.put("key2", "value2_is_longer"); // Stores "key2" -> "value2_is_longer".
store1.put("key3", "value3"); // Stores "key3" -> "value3".
store1.get("key1"); // Returns "value1".
store1.get("key4"); // Returns "".
store1.shutdown(); // Persists the current store and closes this instance.
KVStore store2 = new KVStore(medium); // Creates a new store with same medium.
store2.get("key1"); // Returns "", since nothing has been restored yet.
store2.put("key4", "value4"); // Adds "key4" -> "value4" to store2's in-memory map.
store2.restore(); // Loads the blob, replacing all in-memory data with what was persisted by store1.
store2.get("key1"); // Returns "value1", loaded from persisted state.
store2.get("key2"); // Returns "value2_is_longer".
store2.get("key3"); // Returns "value3".
store2.get("key4"); // Returns "", since "key4" was never persisted and was overwritten by restore.
```


# Persistent Key-Value Store — Python Practice

中文讲解，technical terms 保留 English。按顺序完成 TODO，再运行对应 tests；未完成时出现 `NotImplementedError` 是正常的。

你来写 implementation，这份 notebook 提供练习骨架，不包含完整 solution。

## Requirements

- `put(key, value)` 写入或更新；`get(key)` 在 key 缺失时返回 `""`。
- constructor 创建空 store，不自动 restore。
- `shutdown()` 保存整个 snapshot，成功后 instance 永久 closed；后续 `get()` / `put()` 必须抛出 error。
- `restore()` 用保存的 snapshot **替换**当前 data；missing / empty blob 对应空 store。
- 支持 Unicode、empty strings 和任意 separator 字符。
- 仅通过提供的 `Medium` 读写 blob；禁止 JSON、pickle 和 native object serialization。

```text
in-memory objects → encode → binary blob → persist
binary blob → decode → in-memory objects
```

本练习先假设 saved blob 是完整、合法的，不考虑 concurrent access。


## 1. String encoding helpers

采用 `[4-byte length][UTF-8 payload]`。length 表示 payload 的 **byte count**，使用 unsigned big-endian integer。

```text
"cat" → [00 00 00 03][63 61 74]   (hexadecimal)
""    → [00 00 00 00]             (没有 payload)
```

`(3).to_bytes(4, "big")` 占 4 bytes，表示 integer 3。

可用操作：`text.encode("utf-8")`、`len(...)`、`n.to_bytes(4, "big")`、`int.from_bytes(..., "big")`、`payload.decode("utf-8")`。


In [42]:
def encode_string(text: str) -> bytes:
    """返回 [4-byte length][UTF-8 payload]。"""
    # TODO: encode text，生成 length field，拼接并返回。
    payload=text.encode("utf-8")
    text_len=len(payload).to_bytes(4,"big")
    res=text_len+payload
    return res


def decode_string(blob: bytes, offset: int) -> tuple[str, int]:
    """返回 (decoded text, 下一个未读取的 byte offset)。"""
    # TODO: 读取 length field，移动 offset，读取 payload，再移动 offset。
    # TODO: decode payload，返回 text 和新的 offset。
    text_len=int.from_bytes(blob[offset:offset+4],"big")
    offset+=4

    res=blob[offset:offset+text_len].decode("utf-8")
    offset+=text_len

    print(res, offset)
    return res,offset


In [43]:
# 完成两个 helpers 后运行。
assert encode_string("cat") == b"\x00\x00\x00\x03cat"
assert encode_string("") == b"\x00\x00\x00\x00"

blob = encode_string("你好") + encode_string("")
text, offset = decode_string(blob, 0)
assert (text, offset) == ("你好", 10)
text, offset = decode_string(blob, offset)
assert (text, offset) == ("", 14)
assert offset == len(blob)

for original in ["a", "x#y", "line1\nline2", "\x00", "你好", "🐍", ""]:
    encoded = encode_string(original)
    decoded, end = decode_string(encoded, 0)
    assert decoded == original
    assert end == len(encoded)

print("String helper tests passed")


你好 10
 14
a 5
x#y 7
line1
line2 15
  5
你好 10
🐍 8
 4
String helper tests passed


## 2. Provided Medium

下面是提供给你的 storage abstraction 和用于练习的 implementation，可以直接运行，无需修改。

`Medium` 只保存一个 blob。KVStore 内部保存 Medium reference，不需要长期缓存第二份 blob。


In [44]:
class Medium:
    def save_blob(self, data: bytes) -> None:
        raise NotImplementedError

    def get_blob(self) -> bytes | None:
        raise NotImplementedError


class InMemoryMedium(Medium):
    def __init__(self):
        self._blob: bytes | None = None

    def save_blob(self, data: bytes) -> None:
        self._blob = bytes(data)

    def get_blob(self) -> bytes | None:
        return self._blob


## 3. Implement KVStore

建议按顺序实现：constructor → `_ensure_open()` → `put()` / `get()` → `shutdown()` → `restore()`。

State：`self._data`、`self._medium`、`self._closed`。

整个 snapshot 的 format：

```text
[entry_count: 4 bytes]
[key_byte_length: 4 bytes][key_bytes]
[value_byte_length: 4 bytes][value_bytes]
... repeat entry_count times
```

- `shutdown()` 可用 `bytearray` 和 `.extend()` 逐段构造 blob，复用 `encode_string()`。保存成功后才设置 closed state。
- `restore()` 先读取 entry count，维护 offset，复用 `decode_string()`。
- 将 decoded data 放入新 dict，最后替换当前 dict。
- `restore()` 不能让已经 closed 的 instance 重新变为 open。本练习在新的 open instance 上 restore。

完成基础 methods 后就可以先运行下一格 tests；修改 class 后记得重新运行 class cell。


In [106]:
class KVStore:
    def __init__(self, medium: Medium):
        self.medium=medium
        self._data={} #k:v
        self._closed=False

    def _ensure_open(self) -> None:
        # TODO: closed 时抛出 RuntimeError。
        if self._closed:
            raise RuntimeError

    def put(self, key: str, value: str) -> None:
        # TODO: 检查 state，然后写入 dict。
        if self._closed:
            self._ensure_open()
        self._data[key]=value

    def get(self, key: str) -> str:
        # TODO: 检查 state；missing key 返回 empty string。
        if self._closed:
            self._ensure_open()
        return self._data.get(key,"")

    def shutdown(self) -> None:
        # TODO: encode snapshot，save_blob，成功后关闭 instance。
        self._ensure_open()
        parts=[]
        parts.append(len(self._data).to_bytes(4,"big"))
        for k in self._data:
            v=self._data[k]
            parts.append(encode_string(k))
            parts.append(encode_string(v))
        blobs=b"".join(parts)
        print("blobs",b"".join(parts))
        self.medium.save_blob(blobs)
        self._closed=True

    def restore(self) -> None:
        # TODO: get_blob；处理 missing / empty blob。
        # TODO: decode 到新 dict，然后替换当前 data。
        self._ensure_open()
        self._data={}
        blob = self.medium.get_blob()

        offset=0
        if blob is None:
            return
        if len(blob)<4:
            return
        pairs=int.from_bytes(blob[offset:offset+4],"big")
        offset=4
        new_data={}

        for i in range(pairs):
            key,offset=decode_string(blob,offset)
            value,offset=decode_string(blob,offset)
            new_data[key]=value

        self._data=new_data
        return 




In [60]:
# Stage A: constructor、put、get。
medium = InMemoryMedium()
store = KVStore(medium)
assert store.get("missing") == ""
store.put("a", "first")
store.put("a", "second")
assert store.get("a") == "second"
store.put("", "empty key")
assert store.get("") == "empty key"
store.put("你好", "")
assert store.get("你好") == ""
assert medium.get_blob() is None  # put 不保存 snapshot
print("Basic operations passed")


Basic operations passed


In [92]:
medium = InMemoryMedium()
store = KVStore(medium)
store.put("a", "hello")
store.shutdown()

expected = (1).to_bytes(4, "big") + encode_string("a") + encode_string("hello")
print(medium.get_blob())
print(expected)
assert medium.get_blob() == expected

blobs b'\x00\x00\x00\x01\x00\x00\x00\x01a\x00\x00\x00\x05hello'
b'\x00\x00\x00\x01\x00\x00\x00\x01a\x00\x00\x00\x05hello'
b'\x00\x00\x00\x01\x00\x00\x00\x01a\x00\x00\x00\x05hello'


In [96]:
# Stage B: snapshot、closed state、restore replacement。
medium = InMemoryMedium()
original = {"a": "hello", "你好": "", "": "🐍", "x#y": "line1\nline2\x00"}
writer = KVStore(medium)
for key, value in original.items():
    writer.put(key, value)
writer.shutdown()
assert isinstance(medium.get_blob(), bytes)

for operation in [lambda: writer.get("a"), lambda: writer.put("a", "changed")]:
    try:
        operation()
    except NotImplementedError:
        raise  # TODO 不算通过 closed-state test
    except RuntimeError:
        pass
    else:
        raise AssertionError("Closed instance must reject get/put")

reader = KVStore(medium)
assert reader.get("a") == ""  # constructor 不自动 restore
reader.put("unsaved", "discard me")
reader.put("a", "local change")
reader.restore()
for key, value in original.items():
    assert reader.get(key) == value
assert reader.get("unsaved") == ""

# Public get 无法区分 missing key 与 value=""；检查下一份 snapshot 的 entry count。
reader.shutdown()
snapshot = medium.get_blob()
assert int.from_bytes(snapshot[:4], "big") == len(original)
print("Snapshot and lifecycle tests passed")


blobs b'\x00\x00\x00\x04\x00\x00\x00\x01a\x00\x00\x00\x05hello\x00\x00\x00\x06\xe4\xbd\xa0\xe5\xa5\xbd\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x04\xf0\x9f\x90\x8d\x00\x00\x00\x03x#y\x00\x00\x00\x0cline1\nline2\x00'
a 9
hello 18
你好 28
 32
 36
🐍 44
x#y 51
line1
line2  67
blobs b'\x00\x00\x00\x04\x00\x00\x00\x01a\x00\x00\x00\x05hello\x00\x00\x00\x06\xe4\xbd\xa0\xe5\xa5\xbd\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x04\xf0\x9f\x90\x8d\x00\x00\x00\x03x#y\x00\x00\x00\x0cline1\nline2\x00'
Snapshot and lifecycle tests passed


In [107]:
# Stage C: missing blob、empty blob、empty snapshot。
for saved in [None, b""]:
    medium = InMemoryMedium()
    if saved is not None:
        medium.save_blob(saved)
    store = KVStore(medium)
    store.put("temporary", "discard me")
    store.restore()
    assert store.get("temporary") == ""
    store.shutdown()
    assert medium.get_blob() == (0).to_bytes(4, "big")

medium = InMemoryMedium()
KVStore(medium).shutdown()
reader = KVStore(medium)
reader.put("temporary", "discard me")
reader.restore()
assert reader.get("temporary") == ""
print("Empty-state tests passed")


blobs b'\x00\x00\x00\x00'
blobs b'\x00\x00\x00\x00'
blobs b'\x00\x00\x00\x00'
Empty-state tests passed


## 4. Your scratchpad

在下面添加你自己的 inputs、prints 或 tests。可以用 `list(blob)` 查看每个 byte 的 integer value，用 `blob.hex(" ")` 查看 hexadecimal。


In [ ]:
# Try your own examples here.


## 5. Interview notes — implementation 完成后再讨论

自己填写，之后我们一起 review：

- `N` = entry count，`B` = total UTF-8 payload bytes。各 method 的 time / space complexity：TODO。
- Python dict lookup 的 average complexity 与 string hashing 成本：TODO。
- snapshot 除 payload 外需要多少 metadata bytes：TODO。
- truncated blob、invalid length、invalid UTF-8 如何处理：TODO。
- save_blob 失败时，closed state 应该怎样：TODO。
- 如果 process 在保存中途 crash，Medium 需要提供什么保证：TODO。
- 多个 instances 或 threads 同时访问时，需要怎样定义行为：TODO。
- 数据太大，无法每次保存整个 snapshot 时，可以怎样扩展：TODO。


In [144]:
def encode(text):
    # text -> blob
    # len(text)+text
    byte_text=text.encode("utf-8")
    byte_len=len(byte_text).to_bytes(4,"big")
    return byte_len+byte_text

def decode(blob,offset):
    # blob -> text, offset
    byte_len=blob[offset:offset+4]
    length=int.from_bytes(byte_len,"big")
    offset+=4

    byte_text=blob[offset:offset+length]
    text=byte_text.decode("utf-8")
    offset+=length
    return text,offset

In [145]:
blob=encode("hello")
print(blob)
print(decode(blob,0))

b'\x00\x00\x00\x05hello'
('hello', 9)


In [ ]:
import struct
from abc import ABC, abstractmethod
from io import BytesIO

# The Medium interface allows persisting and retrieving raw binary data.
class Medium(ABC):
    @abstractmethod
    def saveBlob(self, data):
        pass

    @abstractmethod
    def getBlob(self):
        pass

# An in-memory implementation of the Medium interface for testing purposes.
class InMemoryMedium(Medium):
    def __init__(self):
        self.storage = None

    def saveBlob(self, data):
        self.storage = data

    def getBlob(self):
        return self.storage

class KVStore:
    def __init__(self, medium):
        # TODO: Implement __init__ logic.
        self.medium=medium
        self._data={} # key:value
        self.closed=False

    def put(self, key, value):
        # TODO: Implement put logic.
        if self.closed:
            raise RuntimeError
        self._data[key]=value

    def get(self, key):
        # TODO: Implement get logic.
        if self.closed:
            raise RuntimeError
        return self._data.get(key,"")

    def shutdown(self):
        # TODO: Implement shutdown logic.
        if self.closed:
            raise RuntimeError

        #num_pairs + encode(k)+encode(value)
        num_pairs=len(self._data)
        blobs=[num_pairs.to_bytes(4,"big")]
        for k in self._data:
            blobs.append(encode(k))
            value=self._data[k]
            blobs.append(encode(value))

        self.medium.saveBlob(b"".join(blobs))
        self.closed=True
        return 
        

    def restore(self):
        # TODO: Implement restore logic.
        self._data={}

        blobs=self.medium.getBlob()
        if blobs is None or len(blobs)<4:
            return self._data
        num_pairs=int.from_bytes(blobs[:4])
        offset=4
        for i in range(num_pairs):
            key,offset=decode(blobs,offset)
            value,offset=decode(blobs,offset)
            self._data[key]=value
        return self._data
        

    @staticmethod
    def main(args):
        KVStore.test1()
        KVStore.test2()
        KVStore.test3()

    @staticmethod
    def test1():
        print("===== Test 1 =====")

        medium = InMemoryMedium()
        store1 = KVStore(medium)
        store1.put("key1", "value1")
        store1.put("key2", "value2_is_longer")
        store1.put("key3", "value3")
        print(store1.get("key1"))  # Expected: "value1"
        print(store1.get("key4"))  # Expected: ""
        store1.shutdown()

        store2 = KVStore(medium)
        print(store2.get("key1"))  # Expected: ""
        store2.put("key4", "value4")

        store2.restore()
        print(store2.get("key1"))  # Expected: "value1"
        print(store2.get("key2"))  # Expected: "value2_is_longer"
        print(store2.get("key3"))  # Expected: "value3"
        print(store2.get("key4"))  # Expected: ""

    @staticmethod
    def test2():
        print("\n===== Test 2 =====")

        medium = InMemoryMedium()
        store1 = KVStore(medium)
        store1.put("counter", "1")
        store1.put("status", "active")
        print(store1.get("counter"))  # Expected: "1"
        print(store1.get("status"))  # Expected: "active"
        store1.shutdown()

        store2 = KVStore(medium)
        store2.restore()
        store2.put("counter", "2")
        store2.put("status", "inactive")
        store2.put("new_field", "added")
        print(store2.get("counter"))  # Expected: "2"
        print(store2.get("status"))  # Expected: "inactive"
        print(store2.get("new_field"))  # Expected: "added"
        store2.shutdown()

        store3 = KVStore(medium)
        store3.restore()
        print(store3.get("counter"))  # Expected: "2"
        print(store3.get("status"))  # Expected: "inactive"
        print(store3.get("new_field"))  # Expected: "added"

    @staticmethod
    def test3():
        print("\n===== Test 3 =====")

        medium = InMemoryMedium()
        store1 = KVStore(medium)
        store1.put("emoji_key_😊", "emoji_value_🎉")
        store1.put("unicode_key_中文", "unicode_value_日本語")
        store1.put("empty_value", "")
        print(store1.get("emoji_key_😊"))  # Expected: "emoji_value_🎉"
        print(store1.get("empty_value"))  # Expected: ""
        store1.shutdown()

        store2 = KVStore(medium)
        store2.restore()
        print(store2.get("emoji_key_😊"))  # Expected: "emoji_value_🎉"
        print(store2.get("unicode_key_中文"))  # Expected: "unicode_value_日本語"
        print(store2.get("empty_value"))  # Expected: ""
        print(store2.get("nonexistent_key"))  # Expected: ""

if __name__ == "__main__":
    KVStore.main([])

===== Test 1 =====
value1


value1
value2_is_longer
value3


===== Test 2 =====
1
active
2
inactive
added
2
inactive
added

===== Test 3 =====
emoji_value_🎉

emoji_value_🎉
unicode_value_日本語


